# 2 Descriptive (Spatial) Analytics

After completing data preparation in Section 1, the analytical perspective shifts from *"what is in the data"* to *"what does the data tell us"*. This section uses descriptive and exploratory methods to characterize taxi demand patterns in Chicago — without yet constructing any predictive model. The goal is to surface structure, identify spatial hot spots, and develop an empirical understanding of demand dynamics that will directly inform the predictive modeling in Section 3 and the operational recommendations in Section 5.

The main analytical questions guiding this section are:
1. How does demand vary across time (hour, weekday, month, season)?
2. How does demand vary across space, and where are the natural pickup hot spots?
3. How do temporal and spatial patterns interact?
4. Where do trips actually go — what are the dominant origin-destination flows?
5. Which external factors, primarily weather, modulate demand and how strongly?
6. How do individual trip characteristics (length, price, idle time, utilization rate) differ between regions?
7. How sensitive are the observed patterns to the chosen spatial and temporal resolution?

Each of these questions maps back to one of the consulting decisions outlined in Section 1: **strategic** (where to enter the market), **tactical** (when and how many vehicles to deploy), and **operational** (idle time, repositioning, charging). This section is therefore not a descriptive exercise for its own sake, but a structured exploration that delivers evidence for the final recommendations to the client.

Before the analyses begin, a short overview of the central trip variables (trip duration, trip distance, trip total, idle time, utilization rate) is provided. These baseline distributions serve as the reference against which all subsequent spatial and temporal variations are interpreted.

- *2.1 Temporal Patterns*
- *2.2 Spatial Patterns and Hotspot Identification*
- *2.3 Spatio-Temporal Interplay*
- *2.4 Origin-Destination Flows*
- *2.5 External Drivers: Weather*
- *2.6 Spatial Variation in Trip Characteristics*
- *2.7 Resolution Sensitivity*
- *2.8 Summary and Implications*

Sections 2.1 to 2.3 follow a deliberate sequence: first time alone, then space alone, then their joint interaction. This separation makes each marginal effect interpretable before the combined view is presented. Sections 2.4 to 2.6 broaden the perspective to additional dimensions — destination flows, weather effects, and intra-area trip differences — that are conceptually independent of the demand-count framing used so far. Section 2.7 systematically tests how the chosen spatial resolution affects the observed patterns, and Section 2.8 consolidates the findings into operational implications for the fleet operator.

## Structure Rationale

The chapter separates each analytical dimension before recombining them. The order is time → space → joint, then additional dimensions (OD, weather, trip characteristics), then a sensitivity check, then the synthesis.

### 2.1 Temporal Patterns
Demand is aggregated at city level over time. Required outputs:

- Line plot of `demand_count` by hour of day, split into weekday vs. weekend
- Heatmap weekday × hour at city level
- Monthly demand profile over the full sample period
- Time courses of `trip_miles`, `trip_seconds`, `trip_total`, `idle_time`, `utilization_rate`, ... aggregated by hour and weekday

Time is treated first because the temporal cycle is the strongest single driver of demand variation and is well-defined at city level. Once the rhythm is established, spatial effects in 2.2 and 2.3 can be read against it.

### 2.2 Spatial Patterns and Hotspot Identification
Demand is aggregated over all hours and shown spatially through two complementary methods.

Grid-based:
- Choropleth of `demand_count` on Community Areas
- Choropleth of `demand_count` on H3 r? cells (default spatial unit)

Grid-free:
- Gaussian Mixture Model fitted on raw pickup coordinates (latitude, longitude), with the number of components selected via BIC
- Kernel Density Estimation on the same coordinates, plotted as contour map over Chicago

GMM is mandatory per the task description. KDE is added because the cluster centres from GMM do not visualise density gradients well; the contour map fills that gap. The grid-vs-grid-free comparison is the substantive point: if natural pickup clusters do not align with Community Area boundaries, the administrative units are the wrong service-zone proxy.

### 2.3 Spatio-Temporal Interplay
`demand_count` is shown jointly over (spatial unit × time). Three artefacts:

- Heatmap matrix: rows = top-N H3 r? cells (or Community Areas), columns = hour of day, colour = mean `demand_count`
- Small multiples: four to six maps of Chicago at representative time slices (e.g. weekday 1am, 7am, 1pm, 7pm; vs. Saturday?), choropleth on the same spatial unit as 2.2
- Line plot: hourly demand profile for the top five hexagons over a typical day

The three artefacts exist because no single plot captures both dimensions well. The matrix is dense and exact but loses geography; the small multiples retain geography but lose continuity; the line plot makes the dynamics of individual hotspots readable.

### 2.4 Origin-Destination Flows
Pickup → dropoff pairs are aggregated at Community Area level (H3 r? produces too many low-volume pairs for a readable OD analysis). Required outputs:

- Top 20–50 OD pairs as a sorted table with trip count, mean trip distance and mean trip total
- Chord or Sankey diagram of the major flows (I don't even know what that is, but AI says it should be good)
- Asymmetry score per OD pair: `(forward − backward) / (forward + backward)`, used to flag corridors with structural repositioning need

Anchoring on pickups alone misses where vehicles end up after a trip, which is the relevant question for repositioning.

### 2.5 External Drivers: Weather
The hourly weather data from Section 1.5 is merged onto the hourly demand aggregate by `timestamp_h`. Required outputs:

- `demand_count` vs. `temp_h`, binned and plotted with median and IQR per bin
- Same for `precip_h` and `snow_h`
- Comparison of rainy vs. dry days at the same hour of day, to control for the time-of-day confound (rain peaks in the afternoon, demand also peaks in the afternoon — a raw correlation would mostly capture this)
- in generel problem of analyzing temp e.g., because this correlates with time => so try to cancel time-influence and look at data always at 4pm e.g. with different temps. 

Snowfall is analysed only for the winter months; outside winter the signal is too sparse. Weather has only a temporal component, so all comparisons here are city-level.

### 2.6 Spatial Variation in Trip Characteristics
For each Community Area and each H3 r? cell, the following per-area statistics are computed and visualised as choropleths (maybe I am listing too many variables, please recheck):

- mean `trip_miles`, mean `trip_seconds`
- mean `trip_total`, `price_per_mile`, `price_per_minute`
- mean `avg_speed_mph` (operational congestion indicator)
- median `idle_time` per taxi (skewed distribution, so median is used)
- mean `utilization_rate` (active driving time over time on shift)
- mean repositioning distance (Haversine from dropoff to next pickup of the same taxi, averaged within area)
- mean detour factor (`trip_miles` over straight-line pickup–dropoff distance)
- number of distinct companies operating in the area
- share of digital payments (`payment_type` ∈ {Credit Card, Mobile})
- mean tip rate (`tips` / `fare`, conditioned on `fare > 0`)

2.2 counts pickups; 2.6 describes what those pickups produce. Two areas can show the same `demand_count` and produce completely different trip economics, which drives fleet sizing, charging cadence and revenue per pickup.

### 2.7 Resolution Sensitivity
**Spatial sensitivity** (1h aggregate, H3 r4 / r6 / r8 + Community Areas):
- Table per resolution: number of cells, share of (cell × hour) with `demand_count > 0`, median and 99th-percentile demand, Gini coefficient
- Overlay histogram of `demand_count` distributions across resolutions
- Same hour of the same day on a map at three resolutions side by side

Census Tracts are excluded — Section 1.3 documented a 55% missing share, which would dominate any sensitivity effect.

**Temporal sensitivity** (H3 r?, 1h / 4h / 1d): short table on zero-rate and coefficient of variation, plus one example heatmap (cell × time) at the three bin widths. Treated briefly because temporal aggregation is a smoothing operation — variance drops with bin width, but the daily cycle shape does not change.

**GMM vs. grid**: one map with raw pickup points, GMM cluster centres from 2.2 and the top-10 cells at H3 r6 and r8 outlined; short note on the share of pickups captured by each. The point is that GMM is grid-independent, while grid-based hotspots are distorted by cell size — coarse grids merge separate clusters, fine grids split single ones.

Closes with a recommended spatial resolution for the predictive modelling in Section 3, chosen by the trade-off between spatial detail and zero-inflation of the target.

### 2.8 Summary and Implications
The findings from 2.1–2.7 are mapped onto the three consulting questions from Section 1.1:

- Strategic: which areas show consistently high demand across hours and weeks
- Tactical: how demand varies by hour, weekday and weather, with implications for vehicle counts
- Operational: where idle times and repositioning distances are highest, and where the trip mix favours short, high-throughput pickups versus long airport-style trips
